# MMPP FFT Dispersion Interactive Smoke Test

Cel: uruchomić minimalny, powtarzalny test publicznej ścieżki `m.dispersion.plot.interactive()` dla dyspersji FFT na syntetycznych danych `.zarr`.

W tym notebooku `m` oznacza accessor FFT dla datasetu magnetyzacji: `m = job.m.fft`. Dzięki temu komórka testowa używa dokładnie wzorca `m.dispersion.plot.interactive(...)`.


## Co sprawdzamy

- `mmpp.open(...)` znajduje syntetyczny wynik `.zarr`.
- `m.dispersion.plot.interactive(..., show=False)` zwraca lekki kontroler bez uruchamiania UI.
- `compute_1d(..., store_complex=True)` tworzy wynik z kompleksem potrzebnym do modów.
- `res.plot.interactive(show=False)` i `res.modes.interactive(show=False)` działają w ścieżce notebookowej.
- Ostatnia komórka zawiera opcjonalne `show=True` do ręcznego renderu w Jupyterze.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import shutil
import site
import sys
from pathlib import Path

# Musi byc ustawione zanim notebook uruchomi jakiekolwiek instalacje/subprocesy.
os.environ.setdefault("PYTHONNOUSERSITE", "1")

# Matplotlib tworzy cache dopiero przy realnym renderze widgetu; ustawiamy
# lokalizacje zapisywalna, zeby notebook nie zalezal od konfiguracji HOME.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mmpp-matplotlib-cache")

_BINARY_PACKAGES = ["numpy", "pandas", "zarr", "numcodecs", "h5py"]


def _user_site_roots() -> list[Path]:
    roots = []
    for getter in (site.getusersitepackages, site.getsitepackages):
        try:
            value = getter()
        except Exception:
            continue
        values = value if isinstance(value, list) else [value]
        for entry in values:
            try:
                path = Path(entry).resolve()
            except Exception:
                continue
            if ".local" in path.parts or path == Path.home() / ".local":
                roots.append(path)
    return roots


_USER_SITE_ROOTS = _user_site_roots()


def _path_is_under(path: str | None, root: Path) -> bool:
    if not path:
        return False
    try:
        Path(path).resolve().relative_to(root)
    except ValueError:
        return False
    except OSError:
        return False
    return True


def _path_is_under_any(path: str | None, roots: list[Path]) -> bool:
    return any(_path_is_under(path, root) for root in roots)


def _disable_user_site_for_binary_stack() -> dict[str, list[str]]:
    removed_paths = []
    for entry in list(sys.path):
        if _path_is_under_any(entry, _USER_SITE_ROOTS):
            sys.path.remove(entry)
            removed_paths.append(entry)

    preloaded = []
    for name in _BINARY_PACKAGES:
        module = sys.modules.get(name)
        if module is not None and _path_is_under_any(getattr(module, "__file__", None), _USER_SITE_ROOTS):
            preloaded.append(f"{name}: {getattr(module, '__file__', None)}")

    if preloaded:
        raise RuntimeError(
            "Ten kernel juz zaladowal binarne pakiety z ~/.local, a ich nie wolno "
            "bezpiecznie przeladowywac w zywej sesji. W VS Code wybierz kernel "
            "'Python (MMPP no user-site)', kliknij Restart i uruchom notebook od pierwszej "
            "komorki. Zaladowane moduly: "
            + "; ".join(preloaded)
        )

    return {"paths": removed_paths, "preloaded": preloaded}


def _forget_modules(prefixes: list[str]) -> list[str]:
    removed = []
    for name in list(sys.modules):
        if any(name == prefix or name.startswith(prefix + ".") for prefix in prefixes):
            del sys.modules[name]
            removed.append(name)
    return removed


user_site_cleanup = _disable_user_site_for_binary_stack()
if user_site_cleanup["paths"]:
    print({"removed_user_site_paths": user_site_cleanup["paths"]})

# Pozwala uruchomic notebook zarowno z katalogu repo, jak i z output/jupyter-notebook.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "mmpp").is_dir() and (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Nie znaleziono katalogu glownego repo MMPP")

# Bezposrednie ladowanie kodu z checkoutu, bez instalowania paczki.
LOCAL_LIBRARY_PATHS = [REPO_ROOT]
external_libraries = REPO_ROOT / "external_libraries"
if external_libraries.is_dir():
    LOCAL_LIBRARY_PATHS.extend(
        path for path in sorted(external_libraries.iterdir()) if path.is_dir()
    )

for path in reversed(LOCAL_LIBRARY_PATHS):
    path_str = str(path)
    if path_str in sys.path:
        sys.path.remove(path_str)
    sys.path.insert(0, path_str)

# Jesli mmpp bylo juz zaimportowane z bledem core, usuwamy ten stan przed importem.
removed_mmpp_modules = _forget_modules(["mmpp"])
if removed_mmpp_modules:
    print({"reloaded_local_modules": removed_mmpp_modules})


def _import_checked(name: str):
    try:
        module = importlib.import_module(name)
    except ValueError as exc:
        message = str(exc)
        if "numpy.dtype size changed" in message:
            raise RuntimeError(
                "Kernel Jupyter nadal ma niespojne binarne pakiety NumPy/pandas po probie "
                "oczyszczenia user-site. Wykonaj Kernel -> Restart Kernel i uruchom te "
                "komorke jako pierwsza. Najpewniejszy start Jupyter: "
                "PYTHONNOUSERSITE=1 python -m jupyter lab. Oryginalny blad: "
                f"{message}"
            ) from exc
        raise
    return module


def _binary_stack_report() -> dict[str, dict[str, str | None]]:
    report = {"python": {"version": sys.version, "executable": sys.executable}}
    for name in _BINARY_PACKAGES:
        module = _import_checked(name)
        report[name] = {
            "version": getattr(module, "__version__", None),
            "file": getattr(module, "__file__", None),
        }
    return report


binary_stack = _binary_stack_report()
print(json.dumps(binary_stack, indent=2, sort_keys=True))
print({"local_library_paths": [str(path) for path in LOCAL_LIBRARY_PATHS]})

np = _import_checked("numpy")
zarr = _import_checked("zarr")
mmpp = _import_checked("mmpp")

if not getattr(mmpp, "_CORE_AVAILABLE", True):
    raise RuntimeError(
        "mmpp zaimportowal sie bez modulu core. Szczegoly: "
        f"{getattr(mmpp, '_CORE_IMPORT_ERROR', 'unknown')}"
    )

print({"repo": str(REPO_ROOT), "mmpp": mmpp.__version__})


In [ ]:
# Syntetyczna fala spinowa: m_x ~ cos(kx - omega t), m_y ~ sin(kx - omega t).
DATA_DIR = REPO_ROOT / "output" / "jupyter-notebook" / "_tmp_fft_dispersion_interactive"
ZARR_PATH = DATA_DIR / "synthetic-dispersion.zarr"

if ZARR_PATH.exists():
    shutil.rmtree(ZARR_PATH)
DATA_DIR.mkdir(parents=True, exist_ok=True)

n_t, n_x = 32, 64
mode_k_index = 5
mode_f_index = 3

t = np.arange(n_t, dtype=float)[:, None]
x = np.arange(n_x, dtype=float)[None, :]
phase = 2 * np.pi * (mode_k_index * x / n_x - mode_f_index * t / n_t)

data = np.zeros((n_t, 1, 1, n_x, 3), dtype=np.float32)
data[:, 0, 0, :, 0] = 0.15 * np.cos(phase)  # mx
data[:, 0, 0, :, 1] = 0.15 * np.sin(phase)  # my

root = zarr.open(str(ZARR_PATH), mode="w")
root.create_dataset("m", data=data, chunks=data.shape)
root.attrs["t_sampl"] = 1e-12
root.attrs["dx"] = 5e-9
root.attrs["dy"] = 5e-9
root.attrs["description"] = "synthetic fft dispersion interactive smoke"

print({"zarr": str(ZARR_PATH), "shape": data.shape})


In [ ]:
# Publiczna ścieżka użytkownika: baza -> job -> dataset m -> accessor FFT.
db = mmpp.open(str(DATA_DIR), force=True, max_workers=1)
assert len(db) == 1

job = db[0]
dataset_m = job.m
m = dataset_m.fft

print({
    "job": getattr(job, "name", repr(job)),
    "dataset_accessor": type(dataset_m).__name__,
    "fft_accessor": type(m).__name__,
    "has_dispersion": hasattr(m, "dispersion"),
})
assert hasattr(m, "dispersion")


In [ ]:
# Główny smoke test: dokładny wzorzec `m.dispersion.plot.interactive(...)`.
viewer = m.dispersion.plot.interactive(
    axis="x",
    component="mx",
    fmax=800,
    scaling="amplitude_squared",
    disk_cache=False,
    show=False,
)

viewer_state = viewer.state
summary = {
    "viewer_type": type(viewer).__name__,
    "show": viewer_state["show"],
    "positive_frequencies": viewer_state["options"].get("positive_frequencies"),
    "can_reconstruct_modes": viewer_state["can_reconstruct_modes"],
    "notes": viewer_state.get("result_notes", [])[:3],
}
print(json.dumps(summary, indent=2))

assert viewer_state["show"] is False
assert viewer_state["options"].get("positive_frequencies") is True


In [ ]:
# Wynik obliczeniowy i interaktywne widoki result/modes.
res = m.dispersion.compute_1d(
    axis="x",
    component="mx",
    scaling="amplitude_squared",
    store_complex=True,
    disk_cache=False,
)

result_viewer = res.plot.interactive(show=False, fmax=800)
modes_viewer = res.modes.interactive(show=False, lattice_constant_nm=470)

result_summary = {
    "result_shape": list(res.shape),
    "has_complex": res.S_complex is not None,
    "result_viewer_show": result_viewer.state["show"],
    "modes_viewer_show": modes_viewer.state["show"],
    "modes_can_reconstruct": modes_viewer.state["can_reconstruct_modes"],
}
print(json.dumps(result_summary, indent=2))

assert res.S_complex is not None
assert result_viewer.state["show"] is False
assert modes_viewer.state["show"] is False
assert modes_viewer.state["can_reconstruct_modes"] is True


## Opcjonalny render widgetu w Jupyterze

Poniższą komórkę odkomentuj tylko w prawdziwym notebook UI. W trybie automatycznego smoke testu zostaje zakomentowana, żeby test był stabilny headless.


In [ ]:
# live_viewer = m.dispersion.plot.interactive(
#     axis="x",
#     component="mx",
#     fmax=800,
#     scaling="amplitude_squared",
#     disk_cache=False,
#     show=True,
# )
# live_viewer


## Wynik oczekiwany

Jeżeli wszystkie asercje przeszły, publiczna ścieżka `m.dispersion.plot.interactive(..., show=False)` działa na syntetycznym wyniku `.zarr`. Do ręcznej kontroli UI uruchom ostatnią komórkę z `show=True` w Jupyterze z zainstalowanym `ipywidgets` i Matplotlib.
